# Revision 2026-10-02 — Check II: is the PCA baseline wrong by definition?

**Question.** Phase 8 reports that RFM beats PCA (val $|\rho|$ 0.918 vs 0.154 on `valence`, test AUROC 1.000 vs 0.879 overall). `rfm_guppy/baselines.py::pca_vector` mean-centres the difference matrix $D$ before the SVD. Beaglehole et al. (SI Appendix H) do not: they pair the classes at random, stack the differences $a^{(j)}-a^{(i)}$ into $Z$, and take the top eigenvector of $Z$ without centring it. Is the reported PCA gap a property of PCA or an artifact of the centring step?

**Algebra.** Let $D\in\mathbb{R}^{m\times k}$, with rows $D_j = a_{+,j}-a_{-,j}$, $\bar d=\frac1m D^\top\mathbf 1$ and $\tilde D = D-\mathbf 1\bar d^\top$. Then
$$\tilde D^\top\tilde D = D^\top D - m\,\bar d\,\bar d^\top .$$
With $n_+=n_-=m$ (true here: train is balanced, 180/180 for `food` and `env_social`, 60/60 for `valence`), every training point is used exactly once, so $\bar d=\hat\mu_+-\hat\mu_-$, the unnormalised DiffMeans vector. Centring therefore removes exactly the rank-one term along DiffMeans, and the top eigenvector of $\tilde D^\top\tilde D=m\,\widehat{\mathrm{Cov}}(D)$ is a direction of within-class variance. After that, label information enters only through the Pearson sign flip.

**Model / data.** GuppyLM (8.73M), cached last-token residual activations $a_\ell\in\mathbb{R}^{384}$, blocks $\ell=0..5$, concepts `food`, `valence`, `env_social`, train/val/test splits from `data/cache/`. No new inference and no checkpoint needed.

**Variants compared** (same preprocessing for all: centre $Z$ with the train mean, orient by Pearson sign, unit norm):
- `PCA_c` — current code: sequential pairing, $D$ centred.
- `PCA_u` — Appendix H: $D$ **not** centred; sequential pairing, plus random pairing with $B=20$ seeds.
- `DM` — difference of class means (current code).
- RFM — the stored values from `data/cache/<concept>_vectors.npz` and `replication/runs/2026-08-27/baselines_monitoring.json`.

**Falsification criterion (fixed before running).** Uncentred PCA *breaks even* with DiffMeans if, **for all three concepts**, (i) its test AUROC (6-feature aggregate probe, `baseline_monitoring_auroc`) is $\ge$ AUROC$_{\text{DM}}-0.005$, and (ii) its best-block val $|\rho|$ is $\ge |\rho|_{\text{DM}}-0.02$. Both are taken at the worst case over sequential pairing and the 20 random pairings. If PCA breaks even, the claim “RFM is better than PCA” does not hold, because it rests on a mis-specified baseline.

In [1]:
import sys, json, os
import numpy as np, pandas as pd
try: import rfm_guppy.paths
except ModuleNotFoundError:   # repo root not on sys.path: walk up from cwd to the dir holding rfm_guppy/paths.py
    _r = os.getcwd()
    while not os.path.isfile(os.path.join(_r, "rfm_guppy", "paths.py")):
        if _r == os.path.dirname(_r): raise RuntimeError("Run this notebook from inside the repository.")
        _r = os.path.dirname(_r)
    sys.path.insert(0, _r)
from rfm_guppy.paths import ROOT, CACHE_DIR
from rfm_guppy.baselines import (pca_vector, diffmeans_vector, _center, _unit, _orient,
                                 _val_pearson, baseline_monitoring_auroc)
from rfm_checks.checkrun import load_check_run, out_file, replication_run
CACHE = CACHE_DIR
NB = 6; SEEDS = 20
# Run folder: RFM_RUN or the current directory; it must hold a run.json made by `checkrun new`.
RUN_DIR, CFG = load_check_run('II-PCA-baseline', params=dict(NB=NB, SEEDS=SEEDS))
CONCEPTS = CFG['concepts']
REPLICATION_RUN = replication_run(CFG)   # baselines_monitoring.json is read from here

def load(c):
    ds = json.load(open(f'{CACHE}/concept_{c}.json'))
    y = {s: np.array([d['label'] for d in ds[s]], int) for s in ('train', 'val', 'test')}
    A = {s: np.load(f'{CACHE}/{c}_{s}_activations.npy').astype(np.float64) for s in ('train', 'val', 'test')}
    return A, y

DATA = {c: load(c) for c in CONCEPTS}
for c, (A, y) in DATA.items():
    print(c, {s: (A[s].shape, int(y[s].sum())) for s in A})

food {'train': ((360, 6, 384), 180), 'val': ((120, 6, 384), 60), 'test': ((120, 6, 384), 60)}
valence {'train': ((120, 6, 384), 60), 'val': ((40, 6, 384), 20), 'test': ((40, 6, 384), 20)}
env_social {'train': ((360, 6, 384), 180), 'val': ((120, 6, 384), 60), 'test': ((120, 6, 384), 60)}


## 0. Gates
(a) The current `pca_vector` must reproduce the published Phase-8 numbers (best-block val $|\rho|$ from `replication/runs/2026-08-27/baselines_comparison.md`, test AUROC from `replication/runs/2026-08-27/baselines_monitoring.json`). (b) The algebraic identity above must hold numerically, and $\bar d$ must equal $\hat\mu_+-\hat\mu_-$.

In [2]:
def pca_uncentred(Ztr, ytr, Zva, yva, rng=None):
    # Appendix H: pair pos/neg (randomly if rng is given), D = pos - neg, top right singular vector of D, no centring of D.
    Zc, mu = _center(Ztr); Zvc, _ = _center(Zva, mu)
    p, n = np.where(ytr == 1)[0], np.where(ytr == 0)[0]
    if rng is not None: p, n = rng.permutation(p), rng.permutation(n)
    m = min(len(p), len(n)); D = Zc[p[:m]] - Zc[n[:m]]
    v = _orient(_unit(np.linalg.svd(D, full_matrices=False)[2][0]), Zc, ytr)
    return v, mu, _val_pearson(v, Zvc, yva)

def run(method, A, y, **kw):
    V, MU, R = np.zeros((NB, 384)), np.zeros((NB, 384)), np.zeros(NB)
    for b in range(NB):
        V[b], MU[b], R[b] = method(A['train'][:, b], y['train'], A['val'][:, b], y['val'], **kw)
    _, te = baseline_monitoring_auroc(A['train'], y['train'], A['val'], y['val'], A['test'], y['test'], V, MU)
    return dict(V=V, MU=MU, rho=R, auroc=te)

PUB_RHO = {'food': 0.9288, 'valence': 0.1544, 'env_social': 0.4900}
if [c for c in CONCEPTS if c not in PUB_RHO]:   # gate (a) needs the published value; never guess it
    raise ValueError(f'gate (a): PUB_RHO has no published best-block val |rho| of PCA_c for '
                     f'{[c for c in CONCEPTS if c not in PUB_RHO]}; add it from baselines_comparison.md '
                     f'of the replication run that produced the concept')
PUB = json.load(open(os.path.join(REPLICATION_RUN, 'baselines_monitoring.json')))
RES = {}
for c, (A, y) in DATA.items():
    RES[c] = dict(PCA_c=run(pca_vector, A, y), PCA_u=run(pca_uncentred, A, y), DM=run(diffmeans_vector, A, y),
                  PCA_u_rand=[run(pca_uncentred, A, y, rng=np.random.default_rng(s)) for s in range(SEEDS)])
    assert abs(RES[c]['PCA_c']['rho'].max() - PUB_RHO[c]) < 1e-4, c
    assert abs(RES[c]['PCA_c']['auroc'] - PUB[c]['pca']['test_auroc']) < 1e-9, c
    assert abs(RES[c]['DM']['auroc'] - PUB[c]['diffmeans']['test_auroc']) < 1e-9, c
print('gate (a) passed: current pca_vector / diffmeans_vector reproduce the published Phase-8 numbers')

for c, (A, y) in DATA.items():
    for b in range(NB):
        Zc, _ = _center(A['train'][:, b]); p, n = np.where(y['train'] == 1)[0], np.where(y['train'] == 0)[0]
        D = Zc[p] - Zc[n]; m = len(D); dbar = D.mean(0); Dt = D - dbar
        assert np.allclose(Dt.T @ Dt, D.T @ D - m * np.outer(dbar, dbar))
        assert np.allclose(dbar, Zc[y['train'] == 1].mean(0) - Zc[y['train'] == 0].mean(0))
print('gate (b) passed: identity holds and dbar = mu_+ - mu_- in all 18 cells')

gate (a) passed: current pca_vector / diffmeans_vector reproduce the published Phase-8 numbers
gate (b) passed: identity holds and dbar = mu_+ - mu_- in all 18 cells


## A. Per-block comparison
For each block: val $|\rho|$ of each direction, its alignment with DiffMeans, and the share of $\|D\|_F^2$ carried by the rank-one term, $m\|\bar d\|^2/\|D\|_F^2$. That share is what centring throws away.

In [3]:
rows = []
for c, (A, y) in DATA.items():
    R = RES[c]; Vd = R['DM']['V']
    for b in range(NB):
        Zc, _ = _center(A['train'][:, b]); D = Zc[y['train'] == 1] - Zc[y['train'] == 0]
        rows.append(dict(concept=c, block=b, rho_PCA_c=R['PCA_c']['rho'][b], rho_PCA_u=R['PCA_u']['rho'][b],
                         rho_DM=R['DM']['rho'][b],
                         cos_PCA_c_DM=abs(R['PCA_c']['V'][b] @ Vd[b]), cos_PCA_u_DM=abs(R['PCA_u']['V'][b] @ Vd[b]),
                         rank1_share=len(D) * np.sum(D.mean(0) ** 2) / np.sum(D ** 2)))
dfA = pd.DataFrame(rows); print(dfA.round(4).to_string(index=False))

   concept  block  rho_PCA_c  rho_PCA_u  rho_DM  cos_PCA_c_DM  cos_PCA_u_DM  rank1_share
      food      0     0.1554     0.6039  0.8112        0.3451        0.8565       0.1650
      food      1     0.7850     0.9476  0.9531        0.5164        0.9896       0.3338
      food      2     0.8970     0.9591  0.9634        0.5791        0.9921       0.3349
      food      3     0.8863     0.9658  0.9704        0.5172        0.9905       0.3109
      food      4     0.9049     0.9718  0.9749        0.5009        0.9917       0.3013
      food      5     0.9288     0.9719  0.9748        0.5810        0.9906       0.3143
   valence      0     0.0536     0.0297  0.0942        0.7241        0.8187       0.1003
   valence      1     0.1544     0.1936  0.6967        0.2428        0.7620       0.1494
   valence      2     0.1520     0.5708  0.7458        0.1878        0.9539       0.1687
   valence      3     0.0891     0.6022  0.7774        0.1446        0.9514       0.1543
   valence      4    

## B. Headline numbers: best-block val $|\rho|$ and test AUROC
`PCA_u (rand)` reports the median and the worst case over 20 random pairings. RFM numbers are the stored ones.

In [4]:
rows = []
for c in CONCEPTS:
    R = RES[c]; rr = np.array([r['rho'].max() for r in R['PCA_u_rand']]); ra = np.array([r['auroc'] for r in R['PCA_u_rand']])
    rfm_rho = float(np.max(np.load(f'{CACHE}/{c}_vectors.npz')['val_abs_rho']))
    rows.append(dict(concept=c,
        rho_PCA_c=R['PCA_c']['rho'].max(), rho_PCA_u_seq=R['PCA_u']['rho'].max(), rho_PCA_u_rand_med=np.median(rr),
        rho_PCA_u_rand_min=rr.min(), rho_DM=R['DM']['rho'].max(), rho_RFM=rfm_rho,
        auc_PCA_c=R['PCA_c']['auroc'], auc_PCA_u_seq=R['PCA_u']['auroc'], auc_PCA_u_rand_med=np.median(ra),
        auc_PCA_u_rand_min=ra.min(), auc_DM=R['DM']['auroc'], auc_RFM=PUB[c]['rfm']['test_auroc']))
dfB = pd.DataFrame(rows).set_index('concept')
print(dfB.filter(like='rho').round(4).to_string()); print(); print(dfB.filter(like='auc').round(4).to_string())
print('\noverall test AUROC (mean of 3):', dfB[['auc_PCA_c', 'auc_PCA_u_seq', 'auc_PCA_u_rand_min', 'auc_DM']].mean().round(4).to_dict())

            rho_PCA_c  rho_PCA_u_seq  rho_PCA_u_rand_med  rho_PCA_u_rand_min  rho_DM  rho_RFM
concept                                                                                      
food           0.9288         0.9719              0.9718              0.9711  0.9749   0.9782
valence        0.1544         0.6943              0.6976              0.6671  0.7995   0.9183
env_social     0.4900         0.8856              0.8878              0.8824  0.9142   0.9502

            auc_PCA_c  auc_PCA_u_seq  auc_PCA_u_rand_med  auc_PCA_u_rand_min  auc_DM  auc_RFM
concept                                                                                      
food           0.9947          1.000              1.0000              1.0000     1.0      1.0
valence        0.7425          0.995              0.9975              0.9900     1.0      1.0
env_social     0.8992          1.000              1.0000              0.9994     1.0      1.0

overall test AUROC (mean of 3): {'auc_PCA_c': 0.8788, 'auc

## C. Verdict under the pre-fixed criterion

In [5]:
rows = []
for c in CONCEPTS:
    r = dfB.loc[c]
    auc_w = min(r.auc_PCA_u_seq, r.auc_PCA_u_rand_min); rho_w = min(r.rho_PCA_u_seq, r.rho_PCA_u_rand_min)
    ok_auc = auc_w >= r.auc_DM - 0.005; ok_rho = rho_w >= r.rho_DM - 0.02
    rows.append(dict(concept=c, auc_worst=auc_w, auc_DM=r.auc_DM, pass_auc=ok_auc,
                     rho_worst=rho_w, rho_DM=r.rho_DM, gap_rho=r.rho_DM - rho_w, pass_rho=ok_rho, break_even=ok_auc and ok_rho))
dfC = pd.DataFrame(rows); print(dfC.round(4).to_string(index=False))
print('\nBREAK-EVEN on all concepts:', bool(dfC.break_even.all()))
print('AUROC clause alone passes on all concepts:', bool(dfC.pass_auc.all()))

   concept  auc_worst  auc_DM  pass_auc  rho_worst  rho_DM  gap_rho  pass_rho  break_even
      food     1.0000     1.0      True     0.9711  0.9749   0.0038      True        True
   valence     0.9900     1.0     False     0.6671  0.7995   0.1324     False       False
env_social     0.9994     1.0      True     0.8824  0.9142   0.0318     False       False

BREAK-EVEN on all concepts: False
AUROC clause alone passes on all concepts: False


## D. Supplementary (not part of the criterion): is the remaining $|\rho|$ gap larger than sampling noise?
Paired bootstrap over the validation samples (120; only 40 for `valence`), with the directions held fixed at each method's best block. We report 95% intervals for $|\rho|_{\text{RFM}}-|\rho|_{\text{PCA\_u}}$ and $|\rho|_{\text{DM}}-|\rho|_{\text{PCA\_u}}$. This measures only the noise from the val sample; it ignores the noise from training.

In [6]:
from rfm_guppy.rfm import _preprocess
def proj_rfm(c, Zva, b):
    V = np.load(f'{CACHE}/{c}_vectors.npz'); A, y = DATA[c]
    _, mu = _preprocess(A['train'][:, b], True, bool(V['hp_normalize'][b]))
    Zc, _ = _preprocess(Zva, True, bool(V['hp_normalize'][b]), mean_vec=mu)
    return Zc @ V['v_per_block'][b]
def absr(p, y): return abs(np.corrcoef(p, y)[0, 1])

rng = np.random.default_rng(0); NBOOT = 2000; rows = []
for c, (A, y) in DATA.items():
    R = RES[c]; yv = y['val']
    bu, bd = int(np.argmax(R['PCA_u']['rho'])), int(np.argmax(R['DM']['rho']))
    br = int(np.argmax(np.load(f'{CACHE}/{c}_vectors.npz')['val_abs_rho']))
    pu = (A['val'][:, bu] - R['PCA_u']['MU'][bu]) @ R['PCA_u']['V'][bu]
    pd_ = (A['val'][:, bd] - R['DM']['MU'][bd]) @ R['DM']['V'][bd]
    pr = proj_rfm(c, A['val'][:, br], br)
    assert abs(absr(pr, yv) - dfB.loc[c, 'rho_RFM']) < 1e-4
    G = []
    for _ in range(NBOOT):
        i = rng.integers(0, len(yv), len(yv)); G.append((absr(pr[i], yv[i]) - absr(pu[i], yv[i]), absr(pd_[i], yv[i]) - absr(pu[i], yv[i])))
    G = np.array(G)
    rows.append(dict(concept=c, RFM_minus_PCAu=absr(pr, yv) - absr(pu, yv), ci_lo=np.quantile(G[:, 0], .025), ci_hi=np.quantile(G[:, 0], .975),
                     DM_minus_PCAu=absr(pd_, yv) - absr(pu, yv), ci_lo_DM=np.quantile(G[:, 1], .025), ci_hi_DM=np.quantile(G[:, 1], .975)))
dfD = pd.DataFrame(rows); print(dfD.round(4).to_string(index=False))

   concept  RFM_minus_PCAu   ci_lo  ci_hi  DM_minus_PCAu  ci_lo_DM  ci_hi_DM
      food          0.0063 -0.0050 0.0253         0.0030   -0.0032    0.0134
   valence          0.2240  0.0985 0.3951         0.1052    0.0553    0.1698
env_social          0.0645  0.0358 0.0946         0.0286    0.0171    0.0414


## E. Save tables
The summary is in `II-PCA-baseline.md`.

In [7]:
out = lambda name: out_file(os.path.join(RUN_DIR, name))
dfA.to_csv(out('II-PCA-baseline_perblock.csv'), index=False); dfB.to_csv(out('II-PCA-baseline_headline.csv'))
dfC.to_csv(out('II-PCA-baseline_verdict.csv'), index=False); dfD.to_csv(out('II-PCA-baseline_bootstrap.csv'), index=False)
print('saved')

saved
